# COMP34812 NLI — Full Embedding Pre-computation Pipeline

**Author:** Kaan Berk Temizel  
**Track:** Natural Language Inference (NLI)  
**Solution:** Category B — Deep Learning (non-transformer)

## Overview

This notebook **pre-computes and saves all embeddings to disk** for both train and dev splits.
Each token is represented by a **1477-dimensional** vector:

| Component | Dim | Type | Reference |
|---|---|---|---|
| GloVe | 300 | Static, frozen | Pennington et al., 2014 |
| ELMo | 1024 | Contextual, pre-computed | Peters et al., 2018 |
| CharCNN | 100 | Learned | Kim et al., 2016 |
| POS embeddings | 50 | Learned | Universal Dependencies |
| Negation flags | 3 | Rule-based | spaCy dep parse + boundary |
| **Total** | **1477** | | |

## Output Files (saved to Drive)

```
COMP34812_NLI/
├── train_embeddings.pt   — full pre-computed tensors for train split
├── dev_embeddings.pt     — full pre-computed tensors for dev split
└── meta.pt               — vocab, char2idx, pos2idx, glove_matrix
```

Each `.pt` file contains:
```python
{
  'premise_emb':    FloatTensor [N, 64, 1477],  # full concatenated embedding
  'hypothesis_emb': FloatTensor [N, 64, 1477],
  'premise_mask':   LongTensor  [N, 64],
  'hypothesis_mask':LongTensor  [N, 64],
  'labels':         LongTensor  [N],
}
```

## Notebook Structure
1. Environment Setup
2. Data Loading
3. Tokenisation & Vocabulary
4. GloVe Embeddings
5. Character CNN
6. POS Embeddings
7. Negation Flags (3d combined)
8. ELMo Pre-computation (Python 3.10 venv)
9. Full Pre-computation Pipeline
10. Sanity Check
11. Save to Drive

## 1. Environment Setup

In [2]:
# ── Main kernel dependencies ──────────────────────────────────────────────────
!pip install -q spacy psutil
!python -m spacy download en_core_web_sm -q

# ── GloVe weights (Pennington et al., 2014) ───────────────────────────────────
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

# ── ELMo weights (Peters et al., 2018) — original AllenNLP release ───────────
import os
os.makedirs('/content/elmo_model', exist_ok=True)
!wget -q -O /content/elmo_model/options.json \
  https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_options.json
!wget -q -O /content/elmo_model/weights.hdf5 \
  https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5

# ── Python 3.10 venv for AllenNLP ELMo pre-computation ───────────────────────
!pip install -q virtualenv
!virtualenv -p /usr/bin/python3.10 /content/myenv -q
!source /content/myenv/bin/activate && \
  /content/myenv/bin/pip install -q allennlp allennlp-models && \
  /content/myenv/bin/pip install -q 'numpy<2.0' --force-reinstall && \
  /content/myenv/bin/pip install -q torch==1.13.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117 && \
  /content/myenv/bin/pip install -q \
    https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.3.0/en_core_web_sm-3.3.0-py3-none-any.whl

print('Environment setup complete.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 152.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 47.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (p

In [3]:
# ── Main kernel imports ───────────────────────────────────────────────────────
import re
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import spacy
nlp = spacy.load('en_core_web_sm')

# ── Constants ─────────────────────────────────────────────────────────────────
PAD_TOKEN      = '<PAD>'
UNK_TOKEN      = '<UNK>'
MAX_LEN        = 64
MAX_WORD_LEN   = 20
MIN_FREQ       = 2
GLOVE_DIM      = 300
CHAR_EMBED_DIM = 30
CHAR_OUT_DIM   = 100
POS_EMBED_DIM  = 50
ELMO_DIM       = 1024
NEG_DIM        = 3
TOTAL_DIM      = GLOVE_DIM + ELMO_DIM + CHAR_OUT_DIM + POS_EMBED_DIM + NEG_DIM  # 1477

TRIGGERS   = {'not','no','never',"n't",'neither','nor','without','lack',
              'lacking','fail','fails','failed','hardly','barely','scarcely','cannot'}
BOUNDARIES = {',','.','but','and','or','though','although'}

TRAIN_PATH = Path('/content/train.csv')
DEV_PATH   = Path('/content/dev.csv')
GLOVE_PATH = Path('glove.6B.300d.txt')

print(f'Total embedding dim: {TOTAL_DIM}  (1477)')
print('Imports OK')

Total embedding dim: 1477  (1477)
Imports OK


## 2. Data Loading

In [5]:
def load_nli_csv(path):
    df = pd.read_csv(Path(path))
    assert {'premise','hypothesis','label'}.issubset(df.columns)
    before = len(df)
    df = df.dropna(subset=['premise','hypothesis','label']).reset_index(drop=True)
    if before != len(df):
        print(f'  Dropped {before-len(df)} NaN rows')
    df['label'] = df['label'].astype(int)
    print(f'{Path(path).name}: {len(df)} pairs  |  label dist: {dict(df["label"].value_counts().sort_index())}')
    return df

train_df = load_nli_csv(TRAIN_PATH)
dev_df   = load_nli_csv(DEV_PATH)
print(f'\nTotal samples: train={len(train_df)}, dev={len(dev_df)}')

train.csv: 24432 pairs  |  label dist: {0: np.int64(11784), 1: np.int64(12648)}
dev.csv: 6736 pairs  |  label dist: {0: np.int64(3258), 1: np.int64(3478)}

Total samples: train=24432, dev=6736


## 3. Tokenisation & Vocabulary

In [6]:
def tokenise(text):
    """Lowercase tokenise, handles contractions."""
    return re.findall(r"[a-z]+(?:'[a-z]+)*", text.lower().strip())

# tokenise all splits
train_prem_tokens = [tokenise(s) for s in train_df['premise']]
train_hyp_tokens  = [tokenise(s) for s in train_df['hypothesis']]
dev_prem_tokens   = [tokenise(s) for s in dev_df['premise']]
dev_hyp_tokens    = [tokenise(s) for s in dev_df['hypothesis']]

print(f'Sample: {train_df["premise"][0]}')
print(f'Tokens: {train_prem_tokens[0]}')

Sample: yeah i don't know cut California in half or something
Tokens: ['yeah', 'i', "don't", 'know', 'cut', 'california', 'in', 'half', 'or', 'something']


In [7]:
class Vocabulary:
    """
    Word-to-index mapping. Built from training data only — no dev/test leakage.
    Index 0=<PAD>, Index 1=<UNK>. Words below min_freq map to <UNK>.
    """
    def __init__(self, min_freq=MIN_FREQ):
        self.min_freq  = min_freq
        self.word2idx  = {PAD_TOKEN: 0, UNK_TOKEN: 1}
        self.idx2word  = {0: PAD_TOKEN, 1: UNK_TOKEN}
        self.word_freq = Counter()

    def build(self, token_lists):
        for tokens in token_lists:
            self.word_freq.update(tokens)
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
        print(f'Vocabulary: {len(self.word2idx)} tokens  (min_freq={self.min_freq})')

    def encode(self, tokens):
        unk = self.word2idx[UNK_TOKEN]
        return [self.word2idx.get(t, unk) for t in tokens]

    def __len__(self):
        return len(self.word2idx)

vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(train_prem_tokens + train_hyp_tokens)

Vocabulary: 22336 tokens  (min_freq=2)


## 4. GloVe Embeddings

GloVe (Pennington et al., 2014) provides static 300-dimensional word vectors.
Frozen during training. UNK = mean of all GloVe vectors. PAD = zeros.
Words not in GloVe initialised from U(-0.1, 0.1).

In [8]:
def load_glove(glove_path, vocab, embed_dim=GLOVE_DIM):
    vocab_size = len(vocab)
    matrix = np.random.uniform(-0.1, 0.1, (vocab_size, embed_dim)).astype(np.float32)
    matrix[0] = np.zeros(embed_dim)  # PAD
    glove_vecs = {}
    print('Loading GloVe...')
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            glove_vecs[parts[0]] = np.array(parts[1:], dtype=np.float32)
    found = 0
    for word, idx in vocab.word2idx.items():
        if word in glove_vecs:
            matrix[idx] = glove_vecs[word]
            found += 1
    matrix[1] = np.mean(list(glove_vecs.values()), axis=0)  # UNK = mean
    print(f'GloVe: {found}/{vocab_size} words found  |  matrix {matrix.shape}')
    return matrix, glove_vecs

def build_glove_layer(glove_matrix):
    layer = nn.Embedding(len(glove_matrix), GLOVE_DIM, padding_idx=0)
    layer.weight = nn.Parameter(
        torch.tensor(glove_matrix, dtype=torch.float32), requires_grad=False)
    return layer

glove_matrix, glove_vecs = load_glove(GLOVE_PATH, vocab)
glove_layer = build_glove_layer(glove_matrix)
glove_layer.eval()
print('GloVe layer ready (frozen)')

Loading GloVe...
GloVe: 20866/22336 words found  |  matrix (22336, 300)
GloVe layer ready (frozen)


## 5. Character CNN

Character-level CNN (Kim et al., 2016) — 100d per word via three parallel Conv1d
with kernels (2, 3, 4) and filters (33, 33, 34), max-pooled and concatenated.
Captures subword morphology and handles OOV words missed by GloVe.

In [9]:
def build_char_vocab(token_lists):
    char2idx = {'<PAD>': 0, '<UNK>': 1}
    for tokens in token_lists:
        for token in tokens:
            for char in token:
                if char not in char2idx:
                    char2idx[char] = len(char2idx)
    print(f'Char vocab: {len(char2idx)} characters')
    return char2idx

def words_to_char_tensor(token_list, char2idx, max_word_len=MAX_WORD_LEN, max_seq_len=MAX_LEN):
    pad, unk = char2idx['<PAD>'], char2idx['<UNK>']
    result = []
    for i in range(max_seq_len):
        if i < len(token_list):
            word = token_list[i][:max_word_len]
            ids  = [char2idx.get(c, unk) for c in word]
            ids  = ids + [pad] * (max_word_len - len(ids))
        else:
            ids = [pad] * max_word_len
        result.append(ids)
    return torch.tensor(result, dtype=torch.long)  # [max_seq_len, max_word_len]

class CharCNN(nn.Module):
    """
    Character-aware CNN (Kim et al., 2016).
    Embeds characters to 30d, applies three Conv1d (k=2,3,4),
    max-pools each, concatenates → 100d per word.
    """
    def __init__(self, char_vocab_size, char_embed_dim=CHAR_EMBED_DIM, out_dim=CHAR_OUT_DIM):
        super().__init__()
        self.char_embedding = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        nf = out_dim // 3
        self.conv2   = nn.Conv1d(char_embed_dim, nf,     kernel_size=2)
        self.conv3   = nn.Conv1d(char_embed_dim, nf,     kernel_size=3)
        self.conv4   = nn.Conv1d(char_embed_dim, nf + 1, kernel_size=4)
        self.dropout = nn.Dropout(0.3)

    def forward(self, char_ids):
        # char_ids: [batch, seq_len, max_word_len]
        B, S, W = char_ids.shape
        x = self.char_embedding(char_ids.view(B * S, W)).transpose(1, 2)
        def pool(conv): return F.max_pool1d(F.relu(conv(x)), conv(x).shape[-1]).squeeze(-1)
        out = self.dropout(torch.cat([pool(self.conv2), pool(self.conv3), pool(self.conv4)], dim=-1))
        return out.view(B, S, -1)  # [batch, seq_len, 100]

char2idx = build_char_vocab(train_prem_tokens + train_hyp_tokens)
char_cnn  = CharCNN(char_vocab_size=len(char2idx))
char_cnn.eval()
print(f'CharCNN ready  |  output dim: {CHAR_OUT_DIM}')

Char vocab: 29 characters
CharCNN ready  |  output dim: 100


## 6. POS Embeddings

Learned 50-dimensional embeddings for Universal POS tags via spaCy.
POS tags provide explicit syntactic signal complementary to ELMo's implicit encoding.
Vocabulary built from training data only.

In [10]:
def build_pos_vocab(token_lists):
    pos2idx = {'<PAD>': 0, '<UNK>': 1}
    for tokens in token_lists:
        for t in nlp(' '.join(tokens)):
            if t.pos_ not in pos2idx:
                pos2idx[t.pos_] = len(pos2idx)
    print(f'POS vocab: {len(pos2idx)} tags  →  {list(pos2idx.keys())}')
    return pos2idx

def get_pos_ids(token_list, pos2idx, max_len=MAX_LEN):
    unk, pad = pos2idx['<UNK>'], pos2idx['<PAD>']
    ids = [pos2idx.get(t.pos_, unk) for t in nlp(' '.join(token_list))]
    return (ids[:max_len] + [pad] * max_len)[:max_len]

class POSEmbedding(nn.Module):
    """Learned 50d POS tag embeddings."""
    def __init__(self, pos_vocab_size, embed_dim=POS_EMBED_DIM):
        super().__init__()
        self.embedding = nn.Embedding(pos_vocab_size, embed_dim, padding_idx=0)
    def forward(self, pos_ids):
        return self.embedding(pos_ids)

pos2idx       = build_pos_vocab(train_prem_tokens + train_hyp_tokens)
pos_embedding = POSEmbedding(pos_vocab_size=len(pos2idx))
pos_embedding.eval()
print('POS embedding ready')

POS vocab: 19 tags  →  ['<PAD>', '<UNK>', 'INTJ', 'PRON', 'AUX', 'PART', 'VERB', 'PROPN', 'ADP', 'NOUN', 'CCONJ', 'ADJ', 'DET', 'SCONJ', 'ADV', 'NUM', 'X', 'PUNCT', 'SYM']
POS embedding ready


## 7. Negation Flags (3d combined)

Three-dimensional binary feature per token combining two complementary approaches:
- **dim 0** `is_neg`: token is a negation trigger word
- **dim 1** `dep_scope`: token is the head verb of a `neg` dependency (spaCy)
- **dim 2** `boundary_scope`: negation scope active via boundary-based tracking

Motivation: NLI models frequently fail on negation. GloVe has no context awareness.
ELMo captures it implicitly but explicit signal aids the cross-attention mechanism.

In [11]:
def get_negation_flags(token_list, max_len=MAX_LEN):
    """
    Returns [max_len, 3] float32 tensor.
      dim 0: is_neg       — token in TRIGGERS set
      dim 1: dep_scope    — token is head of a neg dependency (spaCy)
      dim 2: boundary_scope — boundary-based scope (resets at BOUNDARIES)
    """
    tokens = [t.lower() for t in token_list[:max_len]]

    # dim 2 — Varun's boundary-based approach
    boundary_flags, in_scope = [], False
    for tok in tokens:
        if tok in TRIGGERS:   in_scope = True
        if tok in BOUNDARIES: in_scope = False
        boundary_flags.append(1.0 if in_scope else 0.0)

    # dim 1 — spaCy dependency scope
    doc = nlp(' '.join(tokens))
    negated_heads = {t.head.i for t in doc if t.dep_ == 'neg'}
    dep_scope = [1.0 if t.i in negated_heads else 0.0 for t in doc]

    # dim 0 — trigger word
    is_neg = [1.0 if t in TRIGGERS else 0.0 for t in tokens]

    flags = []
    for i in range(max_len):
        if i < len(tokens):
            flags.append([
                is_neg[i],
                dep_scope[i] if i < len(dep_scope) else 0.0,
                boundary_flags[i]
            ])
        else:
            flags.append([0.0, 0.0, 0.0])
    return torch.tensor(flags, dtype=torch.float32)  # [max_len, 3]

# quick test
test_flags = get_negation_flags(['the', 'cat', 'is', 'not', 'happy'])
print(f'Negation flags shape: {test_flags.shape}  — expected [64, 3]')
print(f'Token 3 ("not") flags: {test_flags[3]}')

Negation flags shape: torch.Size([64, 3])  — expected [64, 3]
Token 3 ("not") flags: tensor([1., 0., 1.])


## 8. ELMo Pre-computation

ELMo (Peters et al., 2018) produces contextual 1024-dimensional embeddings via a
2-layer BiLSTM over the full sentence. Unlike GloVe, the same word gets different
vectors in different contexts.

**Implementation note:** AllenNLP requires Python ≤ 3.10 and PyTorch < 1.13.
We use a Python 3.10 venv to pre-compute ELMo once and save to disk.
This is consistent with standard practice — Peters et al. recommend pre-computing
ELMo representations as task-specific features for downstream tasks.

In [12]:
# ── Write ELMo pre-computation script ────────────────────────────────────────
elmo_script = '''
import numpy as np
import pandas as pd
import re
import torch
from allennlp.modules.elmo import Elmo, batch_to_ids

TRAIN_PATH   = "/content/train.csv"
DEV_PATH     = "/content/dev.csv"
ELMO_OPTIONS = "/content/elmo_model/options.json"
ELMO_WEIGHTS = "/content/elmo_model/weights.hdf5"
MAX_LEN      = 64
BATCH_SIZE   = 64

print("Loading ELMo (Peters et al., 2018) via AllenNLP...")
elmo   = Elmo(ELMO_OPTIONS, ELMO_WEIGHTS, num_output_representations=1, dropout=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
elmo   = elmo.to(device).eval()
print(f"ELMo loaded on {device}")

def tokenise(text):
    return re.findall(r"[a-z]+(?:[APOS][a-z]+)*", text.lower().strip())

def load_csv(path):
    df = pd.read_csv(path)
    return df["premise"].tolist(), df["hypothesis"].tolist()

def compute_elmo(sentences, name):
    print(f"Computing ELMo: {len(sentences)} {name}...")
    token_lists = [tokenise(s) for s in sentences]
    token_lists = [t[:MAX_LEN] if len(t) > 0 else ["unk"] for t in token_lists]
    all_emb = np.zeros((len(token_lists), MAX_LEN, 1024), dtype=np.float32)
    for i in range(0, len(token_lists), BATCH_SIZE):
        batch    = token_lists[i:i+BATCH_SIZE]
        char_ids = batch_to_ids(batch).to(device)
        with torch.no_grad():
            output = elmo(char_ids)
        emb = output["elmo_representations"][0].cpu().numpy()
        for j in range(len(batch)):
            seq_len = min(len(batch[j]), MAX_LEN)
            all_emb[i+j, :seq_len, :] = emb[j, :seq_len, :]
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {min(i+BATCH_SIZE, len(token_lists))}/{len(token_lists)}")
    print(f"  Done: {all_emb.shape}")
    return all_emb

train_prem, train_hyp = load_csv(TRAIN_PATH)
np.save("/content/elmo_train_prem.npy", compute_elmo(train_prem, "train premises"))
np.save("/content/elmo_train_hyp.npy",  compute_elmo(train_hyp,  "train hypotheses"))
print("Train ELMo saved.")

dev_prem, dev_hyp = load_csv(DEV_PATH)
np.save("/content/elmo_dev_prem.npy", compute_elmo(dev_prem, "dev premises"))
np.save("/content/elmo_dev_hyp.npy",  compute_elmo(dev_hyp,  "dev hypotheses"))
print("Dev ELMo saved.")
print("ALL DONE.")
'''

elmo_script = elmo_script.replace('[APOS]', "['\']")
with open('/content/elmo_precompute.py', 'w') as f:
    f.write(elmo_script)
print('elmo_precompute.py written.')

elmo_precompute.py written.


In [13]:
# ── Run ELMo pre-computation (~15 mins on A100) ───────────────────────────────
# Skip this cell if elmo arrays already exist on Drive
!/content/myenv/bin/python /content/elmo_precompute.py

/content/myenv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/content/myenv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /content/myenv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
Loading ELMo (Peters et al., 2018) via AllenNLP...
ELMo loaded on cuda
Computing ELMo: 24432 train premises...
  64/24432
  704/24432
  1

In [14]:
# ── Load ELMo arrays ─────────────────────────────────────────────────────────
# Option A: from local /content (just pre-computed)
elmo_train_prem = np.load('/content/elmo_train_prem.npy')
elmo_train_hyp  = np.load('/content/elmo_train_hyp.npy')
elmo_dev_prem   = np.load('/content/elmo_dev_prem.npy')
elmo_dev_hyp    = np.load('/content/elmo_dev_hyp.npy')

# Option B: from Drive (after session reset)
# from google.colab import drive; drive.mount('/content/drive')
# d = np.load('/content/drive/MyDrive/COMP34812_NLI/elmo_arrays.npz')
# elmo_train_prem, elmo_train_hyp = d['train_prem'], d['train_hyp']
# elmo_dev_prem,   elmo_dev_hyp   = d['dev_prem'],   d['dev_hyp']

print(f'ELMo shapes:')
print(f'  train prem: {elmo_train_prem.shape}  hyp: {elmo_train_hyp.shape}')
print(f'  dev   prem: {elmo_dev_prem.shape}  hyp: {elmo_dev_hyp.shape}')

ELMo shapes:
  train prem: (24432, 64, 1024)  hyp: (24432, 64, 1024)
  dev   prem: (6736, 64, 1024)  hyp: (6736, 64, 1024)


## 9. Full Pre-computation Pipeline

Pre-compute the full 1477d embedding for every token in every sentence.
This runs once and saves to disk. Training then just loads tensors — no
per-batch computation, reducing training time from ~22h to ~15 minutes.

**Process per sentence:**
1. GloVe lookup → [64, 300]
2. ELMo from pre-computed array → [64, 1024]
3. CharCNN forward → [64, 100]
4. POS embedding forward → [64, 50]
5. Negation flags → [64, 3]
6. Concatenate → [64, 1477]

**Note on learned components (CharCNN, POS):**
These are pre-computed at initialisation with random weights.
They will be **fine-tuned** during training — the saved file stores
the initial representations used as input features.
Varun's OracleNet trains the full pipeline end-to-end.

In [15]:
def get_mask(tokens, max_len=MAX_LEN):
    real = min(len(tokens), max_len)
    return [1] * real + [0] * (max_len - real)

def get_token_ids(tokens, vocab, max_len=MAX_LEN):
    ids = vocab.encode(tokens)[:max_len]
    return ids + [0] * (max_len - len(ids))

@torch.no_grad()
def precompute_split(token_lists_prem, token_lists_hyp,
                     elmo_prem_arr, elmo_hyp_arr,
                     labels, split_name, batch_size=256):
    """
    Pre-computes full 1477d embeddings for all sentences in a split.
    Returns dict with all tensors ready for ResESIM_Dataset.
    """
    N = len(labels)
    print(f'\nPre-computing {split_name}: {N} samples...')

    prem_embs  = torch.zeros(N, MAX_LEN, TOTAL_DIM, dtype=torch.float32)
    hyp_embs   = torch.zeros(N, MAX_LEN, TOTAL_DIM, dtype=torch.float32)
    prem_masks = torch.zeros(N, MAX_LEN, dtype=torch.long)
    hyp_masks  = torch.zeros(N, MAX_LEN, dtype=torch.long)

    char_cnn.eval()
    glove_layer.eval()
    pos_embedding.eval()

    for start in tqdm(range(0, N, batch_size), desc=split_name):
        end = min(start + batch_size, N)
        idx = list(range(start, end))
        B   = len(idx)

        # ── GloVe ─────────────────────────────────────────────────────────────
        prem_ids = torch.tensor(
            [get_token_ids(token_lists_prem[i], vocab) for i in idx], dtype=torch.long)
        hyp_ids  = torch.tensor(
            [get_token_ids(token_lists_hyp[i],  vocab) for i in idx], dtype=torch.long)
        prem_glove = glove_layer(prem_ids)  # [B, 64, 300]
        hyp_glove  = glove_layer(hyp_ids)

        # ── ELMo ──────────────────────────────────────────────────────────────
        prem_elmo = torch.tensor(elmo_prem_arr[start:end], dtype=torch.float32)  # [B, 64, 1024]
        hyp_elmo  = torch.tensor(elmo_hyp_arr[start:end],  dtype=torch.float32)

        # ── CharCNN ───────────────────────────────────────────────────────────
        prem_char = torch.stack(
            [words_to_char_tensor(token_lists_prem[i], char2idx) for i in idx])  # [B, 64, 20]
        hyp_char  = torch.stack(
            [words_to_char_tensor(token_lists_hyp[i],  char2idx) for i in idx])
        prem_char_out = char_cnn(prem_char)  # [B, 64, 100]
        hyp_char_out  = char_cnn(hyp_char)

        # ── POS ───────────────────────────────────────────────────────────────
        prem_pos = torch.tensor(
            [get_pos_ids(token_lists_prem[i], pos2idx) for i in idx], dtype=torch.long)
        hyp_pos  = torch.tensor(
            [get_pos_ids(token_lists_hyp[i],  pos2idx) for i in idx], dtype=torch.long)
        prem_pos_out = pos_embedding(prem_pos)  # [B, 64, 50]
        hyp_pos_out  = pos_embedding(hyp_pos)

        # ── Negation (3d) ─────────────────────────────────────────────────────
        prem_neg = torch.stack(
            [get_negation_flags(token_lists_prem[i]) for i in idx])  # [B, 64, 3]
        hyp_neg  = torch.stack(
            [get_negation_flags(token_lists_hyp[i])  for i in idx])

        # ── Masks ─────────────────────────────────────────────────────────────
        pm = torch.tensor(
            [get_mask(token_lists_prem[i]) for i in idx], dtype=torch.long)
        hm = torch.tensor(
            [get_mask(token_lists_hyp[i])  for i in idx], dtype=torch.long)

        # ── Concatenate → [B, 64, 1477] ───────────────────────────────────────
        prem_full = torch.cat([prem_glove, prem_elmo, prem_char_out, prem_pos_out, prem_neg], dim=-1)
        hyp_full  = torch.cat([hyp_glove,  hyp_elmo,  hyp_char_out,  hyp_pos_out,  hyp_neg],  dim=-1)

        prem_embs[start:end]  = prem_full
        hyp_embs[start:end]   = hyp_full
        prem_masks[start:end] = pm
        hyp_masks[start:end]  = hm

    print(f'  premise_emb:    {prem_embs.shape}')
    print(f'  hypothesis_emb: {hyp_embs.shape}')
    print(f'  Expected dim:   {TOTAL_DIM}  ✓' if prem_embs.shape[-1] == TOTAL_DIM else '  DIMENSION MISMATCH!')

    return {
        'premise_emb':     prem_embs,
        'hypothesis_emb':  hyp_embs,
        'premise_mask':    prem_masks,
        'hypothesis_mask': hyp_masks,
        'labels':          torch.tensor(labels, dtype=torch.long),
    }

print('precompute_split() defined.')

precompute_split() defined.


In [16]:
# ── Pre-compute train split ───────────────────────────────────────────────────
train_data = precompute_split(
    train_prem_tokens, train_hyp_tokens,
    elmo_train_prem, elmo_train_hyp,
    train_df['label'].tolist(),
    'train'
)
print('Train pre-computation complete.')


Pre-computing train: 24432 samples...


train: 100%|██████████| 96/96 [10:49<00:00,  6.77s/it]

  premise_emb:    torch.Size([24432, 64, 1477])
  hypothesis_emb: torch.Size([24432, 64, 1477])
  Expected dim:   1477  ✓
Train pre-computation complete.


In [17]:
# ── Pre-compute dev split ─────────────────────────────────────────────────────
dev_data = precompute_split(
    dev_prem_tokens, dev_hyp_tokens,
    elmo_dev_prem, elmo_dev_hyp,
    dev_df['label'].tolist(),
    'dev'
)
print('Dev pre-computation complete.')


Pre-computing dev: 6736 samples...


dev: 100%|██████████| 27/27 [02:56<00:00,  6.52s/it]

  premise_emb:    torch.Size([6736, 64, 1477])
  hypothesis_emb: torch.Size([6736, 64, 1477])
  Expected dim:   1477  ✓
Dev pre-computation complete.


## 10. Sanity Check

In [18]:
import psutil

print('=== Sanity Check ===')
print(f'Total embedding dim: {TOTAL_DIM}  (expected 1477)')
print()
print('Train:')
for k, v in train_data.items():
    print(f'  {k:20s}: {v.shape}  dtype={v.dtype}')
print()
print('Dev:')
for k, v in dev_data.items():
    print(f'  {k:20s}: {v.shape}  dtype={v.dtype}')

# check no NaN
for split_name, data in [('train', train_data), ('dev', dev_data)]:
    for key in ['premise_emb', 'hypothesis_emb']:
        nan_count = data[key].isnan().sum().item()
        print(f'  {split_name} {key} NaN count: {nan_count}  {"✓" if nan_count == 0 else "⚠ NaN detected!"}')

# label distribution
print(f'\nTrain labels: {train_data["labels"].unique(return_counts=True)}')
print(f'Dev labels:   {dev_data["labels"].unique(return_counts=True)}')

# RAM usage
ram = psutil.virtual_memory()
total_gb = (train_data['premise_emb'].nbytes + train_data['hypothesis_emb'].nbytes +
            dev_data['premise_emb'].nbytes   + dev_data['hypothesis_emb'].nbytes) / 1e9
print(f'\nEmbedding tensors in RAM: {total_gb:.2f} GB')
print(f'Available RAM:            {ram.available/1e9:.1f} GB')
print()
print('Interface to OracleNet:')
print('  logits = oracle_net(batch["premise_emb"], batch["hypothesis_emb"],')
print('                      batch["premise_mask"].sum(1), batch["hypothesis_mask"].sum(1))')

=== Sanity Check ===
Total embedding dim: 1477  (expected 1477)

Train:
  premise_emb         : torch.Size([24432, 64, 1477])  dtype=torch.float32
  hypothesis_emb      : torch.Size([24432, 64, 1477])  dtype=torch.float32
  premise_mask        : torch.Size([24432, 64])  dtype=torch.int64
  hypothesis_mask     : torch.Size([24432, 64])  dtype=torch.int64
  labels              : torch.Size([24432])  dtype=torch.int64

Dev:
  premise_emb         : torch.Size([6736, 64, 1477])  dtype=torch.float32
  hypothesis_emb      : torch.Size([6736, 64, 1477])  dtype=torch.float32
  premise_mask        : torch.Size([6736, 64])  dtype=torch.int64
  hypothesis_mask     : torch.Size([6736, 64])  dtype=torch.int64
  labels              : torch.Size([6736])  dtype=torch.int64
  train premise_emb NaN count: 0  ✓
  train hypothesis_emb NaN count: 0  ✓
  dev premise_emb NaN count: 0  ✓
  dev hypothesis_emb NaN count: 0  ✓

Train labels: (tensor([0, 1]), tensor([11784, 12648]))
Dev labels:   (tensor([0, 1]), 

## 11. Save to Drive

Save three files:
- `train_embeddings.pt` — full pre-computed train tensors
- `dev_embeddings.pt`   — full pre-computed dev tensors
- `meta.pt`             — vocab, char2idx, pos2idx, glove_matrix (for inference)

In [19]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/COMP34812_NLI'
os.makedirs(DRIVE, exist_ok=True)

# ── Save train (compressed numpy) ────────────────────────────────────────────
print('Saving train...')
np.savez_compressed(f'{DRIVE}/train_embeddings.npz',
    premise_emb     = train_data['premise_emb'].numpy(),
    hypothesis_emb  = train_data['hypothesis_emb'].numpy(),
    premise_mask    = train_data['premise_mask'].numpy(),
    hypothesis_mask = train_data['hypothesis_mask'].numpy(),
    labels          = train_data['labels'].numpy(),
)
print(f'train_embeddings.npz: {os.path.getsize(f"{DRIVE}/train_embeddings.npz")/1e9:.2f} GB')

# ── Save dev (compressed numpy) ───────────────────────────────────────────────
print('Saving dev...')
np.savez_compressed(f'{DRIVE}/dev_embeddings.npz',
    premise_emb     = dev_data['premise_emb'].numpy(),
    hypothesis_emb  = dev_data['hypothesis_emb'].numpy(),
    premise_mask    = dev_data['premise_mask'].numpy(),
    hypothesis_mask = dev_data['hypothesis_mask'].numpy(),
    labels          = dev_data['labels'].numpy(),
)
print(f'dev_embeddings.npz:   {os.path.getsize(f"{DRIVE}/dev_embeddings.npz")/1e9:.2f} GB')

# ── Save meta ─────────────────────────────────────────────────────────────────
torch.save({
    'vocab':        vocab,
    'char2idx':     char2idx,
    'pos2idx':      pos2idx,
    'glove_matrix': glove_matrix,
    'total_dim':    TOTAL_DIM,
    'max_len':      MAX_LEN,
}, f'{DRIVE}/meta.pt')
print(f'meta.pt:              {os.path.getsize(f"{DRIVE}/meta.pt")/1e6:.0f} MB')

# ── Save raw ELMo ─────────────────────────────────────────────────────────────
np.savez_compressed(f'{DRIVE}/elmo_arrays.npz',
    train_prem=elmo_train_prem, train_hyp=elmo_train_hyp,
    dev_prem=elmo_dev_prem,     dev_hyp=elmo_dev_hyp)
print(f'elmo_arrays.npz:      {os.path.getsize(f"{DRIVE}/elmo_arrays.npz")/1e9:.2f} GB')

print('\nAll saved. To reload:')
print('  d = np.load(f"{DRIVE}/train_embeddings.npz")')
print('  premise_emb = torch.tensor(d["premise_emb"])')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving train...
train_embeddings.npz: 3.87 GB
Saving dev...
dev_embeddings.npz:   1.06 GB
meta.pt:              41 MB
elmo_arrays.npz:      3.48 GB

All saved. To reload:
  d = np.load(f"{DRIVE}/train_embeddings.npz")
  premise_emb = torch.tensor(d["premise_emb"])


## Usage in ResESIM_Dataset

Once files are saved, training loop loads them with:

```python
from torch.utils.data import Dataset, DataLoader
import torch

class ResESIM_Dataset(Dataset):
    def __init__(self, pt_path):
        data = torch.load(pt_path)
        self.prem_emb  = data['premise_emb']      # [N, 64, 1477].
      
        self.hyp_emb   = data['hypothesis_emb']   # [N, 64, 1477]
        self.prem_mask = data['premise_mask']      # [N, 64]
        self.hyp_mask  = data['hypothesis_mask']  # [N, 64]
        self.labels    = data['labels']            # [N]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'premise_embedding': self.prem_emb[idx],
            'hyp_embedding':     self.hyp_emb[idx],
            'premise_length':    self.prem_mask[idx].sum(),
            'hyp_length':        self.hyp_mask[idx].sum(),
            'label':             self.labels[idx],
        }

train_dataset = ResESIM_Dataset('train_embeddings.pt')
dev_dataset   = ResESIM_Dataset('dev_embeddings.pt')
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader    = DataLoader(dev_dataset,   batch_size=32, shuffle=False)

# OracleNet call per batch:
# logits = oracle_net(
#     batch['premise_embedding'],   # [32, 64, 1477]
#     batch['hyp_embedding'],       # [32, 64, 1477]
#     batch['premise_length'],      # [32]
#     batch['hyp_length'],          # [32]
# )
# INPUT_DIM = 1477, NUM_CLASSES = 2
```

In [23]:
# ── Test round-trip load ──────────────────────────────────────────────────────
d = np.load(f'{DRIVE}/train_embeddings.npz')
print('Keys:', list(d.keys()))

prem = torch.tensor(d['premise_emb'], dtype=torch.float32)
print(f'premise_emb:  {prem.shape}  dtype={prem.dtype}')
print(f'NaN count:    {prem.isnan().sum().item()}')
print(f'Expected:     [24432, 64, 1477]')
print(f'Match: {"✓" if prem.shape == torch.Size([24432, 64, 1477]) else "✗"}')

Keys: ['premise_emb', 'hypothesis_emb', 'premise_mask', 'hypothesis_mask', 'labels']
premise_emb:  torch.Size([24432, 64, 1477])  dtype=torch.float32
NaN count:    0
Expected:     [24432, 64, 1477]
Match: ✓


In [24]:
# ── Full round-trip test: train + dev ─────────────────────────────────────────
from torch.utils.data import DataLoader
import numpy as np
import torch

DRIVE = '/content/drive/MyDrive/COMP34812_NLI'

for split, path, expected_N in [
    ('train', f'{DRIVE}/train_embeddings.npz', 24432),
    ('dev',   f'{DRIVE}/dev_embeddings.npz',    6736),
]:
    print(f'=== {split} ===')
    d = np.load(path)

    prem  = torch.tensor(d['premise_emb'],    dtype=torch.float32)
    hyp   = torch.tensor(d['hypothesis_emb'], dtype=torch.float32)
    pmask = torch.tensor(d['premise_mask'],   dtype=torch.long)
    hmask = torch.tensor(d['hypothesis_mask'],dtype=torch.long)
    labs  = torch.tensor(d['labels'],         dtype=torch.long)

    print(f'  premise_emb:     {prem.shape}')
    print(f'  hypothesis_emb:  {hyp.shape}')
    print(f'  premise_mask:    {pmask.shape}')
    print(f'  hypothesis_mask: {hmask.shape}')
    print(f'  labels:          {labs.shape}')
    print(f'  N correct:       {"✓" if len(labs) == expected_N else f"✗ expected {expected_N}"}')
    print(f'  dim correct:     {"✓" if prem.shape[-1] == 1477 else "✗ expected 1477"}')
    print(f'  NaN prem:        {"✓" if prem.isnan().sum() == 0 else "✗ NaN detected"}')
    print(f'  NaN hyp:         {"✓" if hyp.isnan().sum() == 0 else "✗ NaN detected"}')
    print(f'  label values:    {labs.unique().tolist()}  (expected [0, 1])')
    print(f'  label balance:   {(labs==0).sum().item()} vs {(labs==1).sum().item()}')
    print()

print('Both splits verified. Ready for training.')

=== train ===
  premise_emb:     torch.Size([24432, 64, 1477])
  hypothesis_emb:  torch.Size([24432, 64, 1477])
  premise_mask:    torch.Size([24432, 64])
  hypothesis_mask: torch.Size([24432, 64])
  labels:          torch.Size([24432])
  N correct:       ✓
  dim correct:     ✓
  NaN prem:        ✓
  NaN hyp:         ✓
  label values:    [0, 1]  (expected [0, 1])
  label balance:   11784 vs 12648

=== dev ===
  premise_emb:     torch.Size([6736, 64, 1477])
  hypothesis_emb:  torch.Size([6736, 64, 1477])
  premise_mask:    torch.Size([6736, 64])
  hypothesis_mask: torch.Size([6736, 64])
  labels:          torch.Size([6736])
  N correct:       ✓
  dim correct:     ✓
  NaN prem:        ✓
  NaN hyp:         ✓
  label values:    [0, 1]  (expected [0, 1])
  label balance:   3258 vs 3478

Both splits verified. Ready for training.
